In [8]:
import pandas as pd


data=pd.read_csv("../data/customer-churn-clustered.csv")
print(data.head())

print(data["Churn"].value_counts())
print(data["Cluster"].value_counts())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... TechSupport  \
0  No phone service             DSL             No  ...          No   
1                No             DSL            Yes  ...          No   
2                No             DSL            Yes  ...          No   
3  No phone service             DSL            Yes  ...         Yes   
4                No     Fiber optic             No  ...          No   

  StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No             

In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

y=data["Churn"].map({
    "No":0,
    "Yes":1
})
print(y.value_counts())
X=data.drop(columns=["customerID","Churn"])
# print(x.columns)

# cluster -> str
X["Cluster"]=X["Cluster"].astype(str)

# split
# 
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
    )#la meme proportion de churn dans train et dans test (data desequilibre)

# separation du column
numeric_features=X_train.select_dtypes(
    include=["int64","float64"]
).columns.tolist()

categorical_features=X_train.select_dtypes(
    include=["object"]
).columns.tolist()

# print(numeric_features)
# print(categorical_features)


numeric_transformer = Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])



Churn
0    5174
1    1869
Name: count, dtype: int64


C:\Users\youco\AppData\Local\Temp\ipykernel_16212\3250638415.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features=X_train.select_dtypes(


In [ ]:
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


preprocessor=ColumnTransformer([
    ("num",numeric_transformer,numeric_features),
    ("cat",categorical_transformer,categorical_features)
])

result_with_cluster=[]


def train_evaluate(model, model_name):
    pipeline=ImbPipeline([
        ("preprocessing",preprocessor),
        ("smote",SMOTE(random_state=42)),
        ("model",model)
    ])

    # train
    pipeline.fit(X_train,y_train)

    #prediction
    y_pred_models = pipeline.predict(X_test)
    y_proba_models= pipeline.predict_proba(X_test)[:,1] 
    # metrics
    accuracy = accuracy_score(y_test, y_pred_models)
    precision = precision_score(y_test, y_pred_models,)
    recall = recall_score(y_test, y_pred_models,)
    f1 = f1_score(y_test, y_pred_models,)
    roc_auc = roc_auc_score(y_test, y_proba_models)

    result_with_cluster.append({
        "Modele": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "ROC-AUC": roc_auc
    })

    print(f"\n===== {model_name} avec cluster =====")
    print("Accuracy :", round(accuracy, 4))
    print("Precision :", round(precision, 4))
    print("Recall :", round(recall, 4))
    print("F1-score :", round(f1, 4))
    print("ROC-AUC :", round(roc_auc, 4))

    

    return pipeline



In [23]:
from sklearn.linear_model import LogisticRegression

logistic_with_cluster = train_evaluate(LogisticRegression(max_iter=1000),"Logistic Regression")


===== Logistic Regression avec cluster =====
Accuracy : 0.7381
Precision : 0.5042
Recall : 0.7941
F1-score : 0.6168
ROC-AUC : 0.8403


In [37]:
from sklearn.tree import DecisionTreeClassifier

tree_with_cluster=train_evaluate(DecisionTreeClassifier(random_state=42),"Decision Tree")


===== Decision Tree avec cluster =====
Accuracy : 0.7189
Precision : 0.4743
Recall : 0.5428
F1-score : 0.5062
ROC-AUC : 0.6625


In [24]:
from sklearn.ensemble import RandomForestClassifier

forest_with_cluster = train_evaluate(RandomForestClassifier( n_estimators=100,random_state=42),"Random Forest")


===== Random Forest avec cluster =====
Accuracy : 0.7601
Precision : 0.5503
Recall : 0.5267
F1-score : 0.5383
ROC-AUC : 0.8109


In [25]:
from sklearn.svm import SVC

svm_with_cluster = train_evaluate(
    SVC(probability=True,random_state=42),
    "SVM"
)

c:\CustomerRisk\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



===== SVM avec cluster =====
Accuracy : 0.7566
Precision : 0.5305
Recall : 0.7219
F1-score : 0.6116
ROC-AUC : 0.821


In [26]:
from xgboost import XGBClassifier

xgb_with_cluster = train_evaluate(XGBClassifier(n_estimators=100,random_state=42,eval_metric="logloss"),
    "XGBoost"
)


===== XGBoost avec cluster =====
Accuracy : 0.7786
Precision : 0.5807
Recall : 0.5963
F1-score : 0.5884
ROC-AUC : 0.8162


In [27]:
from sklearn.ensemble import GradientBoostingClassifier

gb_with_cluster = train_evaluate(
    GradientBoostingClassifier(random_state=42),
    "Gradient Boosting"
)


===== Gradient Boosting avec cluster =====
Accuracy : 0.7757
Precision : 0.5656
Recall : 0.6684
F1-score : 0.6127
ROC-AUC : 0.8398


In [ ]:
results_with_cluster_df = pd.DataFrame(result_with_cluster)

results_with_cluster_df = results_with_cluster_df.sort_values(
    by="Recall",
    ascending=False
)
results_with_cluster_df

,Modele,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistic Regression,0.738112,0.504244,0.794118,0.616822,0.840303
2,SVM,0.756565,0.530452,0.721925,0.611552,0.821033
4,Gradient Boosting,0.775727,0.565611,0.668449,0.612745,0.839827
3,XGBoost,0.778566,0.580729,0.596257,0.588391,0.816175
1,Random Forest,0.760114,0.550279,0.526738,0.538251,0.810866


In [29]:
results_with_cluster_df.to_csv(
    "../results/classification_with_cluster.csv",
    index=False
)

In [35]:
results_without = pd.read_csv(
    "../results/classification_without_cluster.csv"
)

results_with = pd.read_csv(
    "../results/classification_with_cluster.csv"
)

comparison = results_without.merge(
    results_with,
    on="Modele",
    suffixes=("_sans_cluster", "_avec_cluster")
)
comparison["Evolution_Recall"] = (
    comparison["Recall_avec_cluster"]
    - comparison["Recall_sans_cluster"]
)

comparison


,Modele,Accuracy_sans_cluster,Precision_sans_cluster,Recall_sans_cluster,F1-score_sans_cluster,ROC-AUC_sans_cluster,Accuracy_avec_cluster,Precision_avec_cluster,Recall_avec_cluster,F1-score_avec_cluster,ROC-AUC_avec_cluster,Evolution_Recall
0,Logistic Regression,0.737402,0.503390,0.794118,0.616183,0.841639,0.738112,0.504244,0.794118,0.616822,0.840303,0.000000
1,Random Forest,0.774308,0.577778,0.556150,0.566757,0.822514,0.760114,0.550279,0.526738,0.538251,0.810866,-0.029412
2,SVM,0.754436,0.527344,0.721925,0.609481,0.821000,0.756565,0.530452,0.721925,0.611552,0.821033,0.000000
3,XGBoost,0.732434,0.497317,0.743316,0.595927,0.842302,0.778566,0.580729,0.596257,0.588391,0.816175,-0.147059
4,Gradient Boosting,0.773598,0.560706,0.679144,0.614268,0.841117,0.775727,0.565611,0.668449,0.612745,0.839827,-0.010695


In [38]:
import joblib
import os

joblib.dump(
    logistic_with_cluster,
    "../models/logistic_with_cluster.pkl"
)

joblib.dump(
    tree_with_cluster,
    "../models/decision_tree_with_cluster.pkl"
)

joblib.dump(
    forest_with_cluster,
    "../models/random_forest_with_cluster.pkl"
)

joblib.dump(
    svm_with_cluster,
    "../models/svm_with_cluster.pkl"
)

joblib.dump(
    xgb_with_cluster,
    "../models/xgboost_with_cluster.pkl"
)

joblib.dump(
    gb_with_cluster,
    "../models/gradient_boosting_with_cluster.pkl"
)

# print(logistic_with_cluster)

['../models/gradient_boosting_with_cluster.pkl']